# PredictShield AI - Improved Training (Patent-Level)

This notebook fixes the two weaknesses from the baseline run:

1. **Rare attack classes** (XSS, SQL Injection, Web Brute Force, Heartbleed, Infiltration) had poor F1 -> fix with **SMOTE oversampling + XGBoost sample weights**
2. **Isolation Forest** had only 30% attack recall -> fix with **contamination tuning + decision_function threshold optimization**

Both models get **full metrics** (accuracy, per-class F1, confusion matrix) this time.

In [1]:
import time
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score,
    confusion_matrix, precision_score, recall_score, roc_auc_score
)
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

print('imports OK')

imports OK


In [2]:
CSV_PATH = '/home/pritam/Desktop/Python/HOMEProject/cicids2017_training_ready.csv'
RANDOM_STATE = 42

t0 = time.time()
df = pd.read_csv(CSV_PATH, dtype=np.float32)
df['label'] = df['label'].astype(np.int32)
print(f'Loaded {df.shape[0]:,} rows x {df.shape[1]} cols in {time.time()-t0:.1f}s')

Loaded 2,827,876 rows x 78 cols in 16.6s


## Step 1: Downsample BENIGN (moderate)

Keep BENIGN at 200k (enough normal baseline), keep ALL attack rows.

In [3]:
BENIGN_TARGET = 200_000

benign = df[df['label'] == 0]
attacks = df[df['label'] != 0]

benign_sampled = benign.sample(n=BENIGN_TARGET, random_state=RANDOM_STATE)
balanced = pd.concat([benign_sampled, attacks], axis=0).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'Downsampled dataset: {len(balanced):,} rows')
del df, benign, attacks, benign_sampled

Downsampled dataset: 756,556 rows


## Step 2: SMOTE oversampling for rare classes

Rare classes (Heartbleed=11, Infiltration=36, SQL=21, XSS=652, Web Brute Force=1507, Bot=1956) get oversampled to a minimum of 5000 each. This directly fixes the weak F1 from the baseline.

In [4]:
X = balanced.drop(columns=['label']).astype(np.float32)
y = balanced['label'].astype(np.int32)

# Classes with fewer than 5000 samples get oversampled to 5000
MIN_SAMPLES = 5000
counts = y.value_counts().sort_index()
smote_strategy = {}
for cls, cnt in counts.items():
    if cls != 0 and cnt < MIN_SAMPLES:
        smote_strategy[int(cls)] = MIN_SAMPLES

print('SMOTE oversampling strategy (class -> target):')
for k, v in smote_strategy.items():
    print(f'  class {k:2d}: {counts[k]:>6d} -> {v}')

smote = SMOTE(sampling_strategy=smote_strategy, random_state=RANDOM_STATE, k_neighbors=3)
X_res, y_res = smote.fit_resample(X, y)

print(f'\nAfter SMOTE: {X_res.shape[0]:,} rows')
print(y_res.value_counts().sort_index().to_string())

del X, y, balanced

SMOTE oversampling strategy (class -> target):
  class  1:   1956 -> 5000
  class  8:     11 -> 5000
  class  9:     36 -> 5000
  class 12:   1507 -> 5000
  class 13:     21 -> 5000
  class 14:    652 -> 5000

After SMOTE: 782,373 rows
label
0     200000
1       5000
2     128025
3      10293
4     230124
5       5499
6       5796
7       7935
8       5000
9       5000
10    158804
11      5897
12      5000
13      5000
14      5000


In [5]:
# Stratified 70/15/15 split
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X_res, y_res, test_size=0.30, stratify=y_res, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}')
del X_res, y_res, X_tmp, y_tmp

Train: 547,661  Val: 117,356  Test: 117,356


## Step 3: XGBoost with class weights + lower learning rate

- `sample_weight` inversely proportional to class frequency (helps rare classes)
- `learning_rate=0.05` + `n_estimators=400` (slower but better convergence)
- `early_stopping_rounds=30`

In [6]:
# Compute sample weights: rare classes get higher weight
class_counts = pd.Series(y_train).value_counts()
class_weights = {c: len(y_train) / (len(class_counts) * cnt) for c, cnt in class_counts.items()}
sample_w = pd.Series(y_train).map(class_weights).values

print('Class weights (higher = rarer class gets more importance):')
for c in sorted(class_weights):
    print(f'  class {c:2d}: weight {class_weights[c]:.3f}  (n={class_counts[c]:,})')

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    objective='multi:softprob',
    num_class=15,
    eval_metric='mlogloss',
    tree_method='hist',
    n_jobs=-1,
    early_stopping_rounds=30,
    random_state=RANDOM_STATE,
)

t0 = time.time()
xgb.fit(
    X_train, y_train,
    sample_weight=sample_w,
    eval_set=[(X_val, y_val)],
    verbose=False,
)
print(f'\nXGBoost trained in {time.time()-t0:.1f}s (best_iteration={xgb.best_iteration})')

Class weights (higher = rarer class gets more importance):
  class  0: weight 0.261  (n=140,000)
  class  1: weight 10.432  (n=3,500)
  class  2: weight 0.407  (n=89,617)
  class  3: weight 5.067  (n=7,205)
  class  4: weight 0.227  (n=161,087)
  class  5: weight 9.486  (n=3,849)
  class  6: weight 8.999  (n=4,057)
  class  7: weight 6.573  (n=5,555)
  class  8: weight 10.432  (n=3,500)
  class  9: weight 10.432  (n=3,500)
  class 10: weight 0.328  (n=111,163)
  class 11: weight 8.845  (n=4,128)
  class 12: weight 10.432  (n=3,500)
  class 13: weight 10.432  (n=3,500)
  class 14: weight 10.432  (n=3,500)

XGBoost trained in 1798.6s (best_iteration=399)


In [7]:
# Full evaluation
y_pred = xgb.predict(X_test)
y_proba = xgb.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')

print(f'XGBoost Accuracy : {acc:.4f}')
print(f'XGBoost F1 (macro): {f1_macro:.4f}')
print(f'XGBoost F1 (weighted): {f1_weighted:.4f}')
print()
print(classification_report(y_test, y_pred, digits=3))

XGBoost Accuracy : 0.9969
XGBoost F1 (macro): 0.9733
XGBoost F1 (weighted): 0.9969

              precision    recall  f1-score   support

           0      1.000     0.998     0.999     30000
           1      0.968     0.996     0.982       750
           2      1.000     1.000     1.000     19204
           3      0.995     0.999     0.997      1544
           4      0.999     1.000     1.000     34519
           5      0.993     0.996     0.995       825
           6      0.994     0.994     0.994       869
           7      1.000     0.999     1.000      1190
           8      1.000     1.000     1.000       750
           9      1.000     0.999     0.999       750
          10      0.999     0.999     0.999     23821
          11      1.000     1.000     1.000       884
          12      0.856     0.768     0.810       750
          13      0.999     0.999     0.999       750
          14      0.785     0.875     0.827       750

    accuracy                          0.997    117

## Step 4: Isolation Forest with threshold optimization

Isolation Forest is unsupervised, so instead of relying on the default `predict()` (which uses contamination=0.05), we:

1. Train on BENIGN-only traffic
2. Use `decision_function()` (raw anomaly score) and sweep a threshold
3. Pick the threshold that maximizes F1 on the VALIDATION set
4. Apply that threshold on the TEST set

This gives a proper accuracy / F1 / confusion matrix.

In [8]:
# Train on BENIGN only (label 0)
X_benign = X_train[y_train == 0]
print(f'BENIGN training rows for IF: {len(X_benign):,}')

iso = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    max_samples=0.5,
)

t0 = time.time()
iso.fit(X_benign)
print(f'Isolation Forest trained in {time.time()-t0:.1f}s')

BENIGN training rows for IF: 140,000
Isolation Forest trained in 13.1s


In [9]:
# Sweep thresholds on VALIDATION set to maximize F1
# decision_function returns positive for normal, negative for anomaly
val_scores = iso.decision_function(X_val)
y_val_binary = (y_val != 0).astype(int)  # 1 = attack, 0 = benign

best_threshold = 0.0
best_f1 = 0.0

print('Threshold sweep (val):')
for thr in np.arange(-0.20, 0.25, 0.01):
    pred = (val_scores < thr).astype(int)  # below threshold = anomaly
    f1 = f1_score(y_val_binary, pred, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thr

print(f'Best threshold: {best_threshold:.3f}')
print(f'Best val F1: {best_f1:.4f}')

Threshold sweep (val):
Best threshold: 0.090
Best val F1: 0.9067


In [10]:
# Apply best threshold on TEST set -> full metrics
test_scores = iso.decision_function(X_test)
y_test_binary = (y_test != 0).astype(int)
if_pred = (test_scores < best_threshold).astype(int)

acc = accuracy_score(y_test_binary, if_pred)
f1 = f1_score(y_test_binary, if_pred)
prec = precision_score(y_test_binary, if_pred)
rec = recall_score(y_test_binary, if_pred)

print('Isolation Forest (threshold-tuned) results:')
print(f'  Accuracy : {acc:.4f}')
print(f'  Precision: {prec:.4f}')
print(f'  Recall   : {rec:.4f}')
print(f'  F1       : {f1:.4f}')
print()
print('Confusion matrix (rows=true, cols=pred):')
print(confusion_matrix(y_test_binary, if_pred))

Isolation Forest (threshold-tuned) results:
  Accuracy : 0.8452
  Precision: 0.8278
  Recall   : 1.0000
  F1       : 0.9058

Confusion matrix (rows=true, cols=pred):
[[11832 18168]
 [    1 87355]]


## Step 5: Save improved models

In [11]:
OUT_DIR = '/home/pritam/Desktop/Python/HOMEProject/Training Files'

joblib.dump(xgb, f'{OUT_DIR}/xgboost_cicids2017_v2.pkl')
joblib.dump(iso, f'{OUT_DIR}/isolation_forest_cicids2017_v2.pkl')

# Save the IF threshold too (needed at inference time)
joblib.dump({'threshold': best_threshold}, f'{OUT_DIR}/isolation_forest_threshold.pkl')

print('Improved models saved:')
print(f'  {OUT_DIR}/xgboost_cicids2017_v2.pkl')
print(f'  {OUT_DIR}/isolation_forest_cicids2017_v2.pkl')
print(f'  {OUT_DIR}/isolation_forest_threshold.pkl')

Improved models saved:
  /home/pritam/Desktop/Python/HOMEProject/Training Files/xgboost_cicids2017_v2.pkl
  /home/pritam/Desktop/Python/HOMEProject/Training Files/isolation_forest_cicids2017_v2.pkl
  /home/pritam/Desktop/Python/HOMEProject/Training Files/isolation_forest_threshold.pkl


## Summary

**Improvements over baseline:**
- XGBoost: SMOTE + class weights -> rare classes (XSS/SQL/Heartbleed/Infiltration) F1 should jump significantly
- Isolation Forest: threshold optimization -> recall should jump from 30% to much higher
- Both models now have full accuracy/F1/confusion matrix metrics